# Pulsatile Membrane Chamber — Numerical Work

This notebook is a guided, code-first foundation for a two-dimensional fluid chamber driven by a prescribed oscillating membrane. Work through the cells in order and read the assumptions before interpreting the numerical results.

**Scope:** this notebook verifies prescribed membrane kinematics, a periodic incompressible pressure projection, and immersed-boundary kernel transfer. It is not yet a complete coupled Navier–Stokes solver with a moving internal wall. In particular, the periodic projection test below is a numerical building block, not the final chamber boundary-condition treatment.

## Learning sequence

1. Define the model and prescribed membrane motion.
2. Visualise the membrane's position and velocity.
3. Understand the pressure-projection equation.
4. Test a divergence-free projection on a periodic grid.
5. spread boundary forces with a regularised delta kernel and check force conservation.
6. Interpolate grid velocity to marker points.
7. Estimate time-step limits and plan validation.


## 1. Model assumptions

Begin with a two-dimensional, incompressible Newtonian fluid with constant density $\rho$ and dynamic viscosity $\mu$. The fluid is stored on a fixed Eulerian grid. The membrane is represented by moving Lagrangian markers $\mathbf{X}(s,t)$.

For the first stage, prescribe the membrane motion rather than predicting it from elasticity:

```math
y_m(t)=y_0+A\sin(2\pi f t)
```

Differentiating with respect to time gives its vertical velocity:

```math
v_m(t)=\frac{dy_m}{dt}=2\pi f A\cos(2\pi f t)
```

Here $y_0$ is the mean position, $A$ is the amplitude, and $f$ is the frequency in hertz. Check units: $y_0$ and $A$ have units of length, and $v_m$ has units of length per time.

A prescribed motion is useful because it separates the question “does the fluid solver respond correctly?” from the harder question “does a fluid-coupled elastic membrane move correctly?”

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Illustrative, dimensionally consistent parameters.
y0 = 0.50       # mean membrane position [m]
amplitude = 0.05  # displacement amplitude [m]
frequency = 1.0   # oscillation frequency [Hz]

period = 1.0 / frequency
t = np.linspace(0.0, 2.0 * period, 500)

y_membrane = y0 + amplitude * np.sin(2.0 * np.pi * frequency * t)
v_membrane = 2.0 * np.pi * frequency * amplitude * np.cos(
    2.0 * np.pi * frequency * t
)

fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
axes[0].plot(t, y_membrane)
axes[0].set_ylabel("Membrane position y [m]")
axes[0].grid(True)
axes[1].plot(t, v_membrane)
axes[1].set_xlabel("Time [s]")
axes[1].set_ylabel("Membrane velocity [m/s]")
axes[1].grid(True)
plt.tight_layout()
plt.show()

### Checkpoint

- At what time is the membrane at its highest position?
- At what time is its vertical velocity zero?
- Why is the velocity proportional to cosine rather than sine?

At a turning point, displacement is at an extreme and instantaneous velocity is zero. This gives a simple analytic check for any later implementation.

## 2. The incompressible fluid equations

The momentum equation for an incompressible Newtonian fluid with a force density $\mathbf{F}$ is

```math
\rho\left(
\frac{\partial\mathbf{u}}{\partial t}
+(\mathbf{u}\cdot\nabla)\mathbf{u}
\right)
=-\nabla p+\mu\nabla^2\mathbf{u}+\mathbf{F}
```

Incompressibility adds the constraint

```math
\nabla\cdot\mathbf{u}=0
```

The force density may include force spread from the immersed membrane. Pressure is not updated independently like an ordinary state variable: in a projection method it is solved to correct the predicted velocity so the incompressibility constraint is satisfied.

## 3. Pressure projection

First predict a velocity $\mathbf{u}^*$ using the momentum effects, then correct it using pressure:

```math
\mathbf{u}^{n+1}=\mathbf{u}^*-\frac{\Delta t}{\rho}\nabla p
```

Take the divergence and require $\nabla\cdot\mathbf{u}^{n+1}=0$:

```math
\nabla^2p=\frac{\rho}{\Delta t}\nabla\cdot\mathbf{u}^*
```

This is the pressure Poisson equation. The precise discrete gradient, divergence, Laplacian, and pressure boundary conditions must be mutually compatible. In the next cell, we use a spectral projection on a periodic box because it provides a compact test of the underlying idea. A periodic box is not the final physical chamber geometry.

In [ ]:
# Spectral projection test on a periodic square domain.
# This cell tests pressure correction only; it is not a full Navier–Stokes time step.

nx = ny = 64
Lx = Ly = 1.0
dx, dy = Lx / nx, Ly / ny
dt = 0.001
rho = 1000.0

x = np.arange(nx) * dx
y = np.arange(ny) * dy
Xg, Yg = np.meshgrid(x, y, indexing="xy")

# Arbitrary smooth periodic predictor field, not necessarily divergence-free.
u_star = np.sin(2 * np.pi * Xg / Lx) + 0.25 * np.cos(2 * np.pi * Yg / Ly)
v_star = 0.4 * np.cos(2 * np.pi * Xg / Lx) + np.sin(2 * np.pi * Yg / Ly)

kx = 2 * np.pi * np.fft.fftfreq(nx, d=dx)
ky = 2 * np.pi * np.fft.fftfreq(ny, d=dy)
KX, KY = np.meshgrid(kx, ky, indexing="xy")
K2 = KX**2 + KY**2

u_hat = np.fft.fft2(u_star)
v_hat = np.fft.fft2(v_star)

# Fourier representation of divergence: i*kx*u_hat + i*ky*v_hat.
div_hat = 1j * KX * u_hat + 1j * KY * v_hat

# Solve -|k|^2 p_hat = (rho/dt) div_hat.
p_hat = np.zeros_like(div_hat, dtype=complex)
nonzero = K2 > 0
p_hat[nonzero] = -(rho / dt) * div_hat[nonzero] / K2[nonzero]
p_hat[0, 0] = 0.0  # pressure's arbitrary constant is set to zero

# Correct velocity in Fourier space.
u_new_hat = u_hat - (dt / rho) * (1j * KX * p_hat)
v_new_hat = v_hat - (dt / rho) * (1j * KY * p_hat)
u_new = np.fft.ifft2(u_new_hat).real
v_new = np.fft.ifft2(v_new_hat).real

div_before = np.fft.ifft2(div_hat).real
div_after_hat = 1j * KX * u_new_hat + 1j * KY * v_new_hat
div_after = np.fft.ifft2(div_after_hat).real

print(f"Maximum |divergence| before projection: {np.max(np.abs(div_before)):.3e}")
print(f"Maximum |divergence| after projection:  {np.max(np.abs(div_after)):.3e}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im0 = axes[0].imshow(div_before, origin="lower", extent=[0, Lx, 0, Ly])
axes[0].set_title("Divergence before projection")
plt.colorbar(im0, ax=axes[0], shrink=0.8)
im1 = axes[1].imshow(div_after, origin="lower", extent=[0, Lx, 0, Ly])
axes[1].set_title("Divergence after projection")
plt.colorbar(im1, ax=axes[1], shrink=0.8)
for ax in axes:
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
plt.tight_layout()
plt.show()

### What this test does and does not show

It demonstrates that pressure can remove the divergent component of a velocity field under periodic spectral differentiation. It does **not** show that a chamber with solid walls or an internal membrane has been solved correctly. The wall conditions and the fluid–membrane coupling remain separate tasks.

A useful validation metric is the maximum absolute divergence before and after projection. The post-projection value should be close to floating-point error for this spectral test.

## 4. Regularised delta kernel

The immersed-boundary method connects Lagrangian markers to the Eulerian grid using a regularised delta function. Force spreading is

```math
\mathbf{F}(\mathbf{x},t)=\int \mathbf{f}(s,t)\,\delta_h(\mathbf{x}-\mathbf{X}(s,t))\,ds
```

Velocity interpolation is

```math
\mathbf{U}(s,t)=\int \mathbf{u}(\mathbf{x},t)\,\delta_h(\mathbf{x}-\mathbf{X}(s,t))\,d\mathbf{x}
```

For the four-point Peskin kernel, define a one-dimensional function $\phi(r)$ by

```math
\phi(r)=
\begin{cases}
\frac{1}{8}\left(3-2|r|+\sqrt{1+4|r|-4r^2}\right), & |r|\leq 1,\\
\frac{1}{8}\left(5-2|r|-\sqrt{-7+12|r|-4r^2}\right), & 1<|r|\leq 2,\\
0, & |r|>2.
\end{cases}
```

On a uniform grid, the two-dimensional kernel is the product of the one-dimensional kernels, each scaled by its grid spacing. This is a numerical regularisation, not an infinitely sharp point delta.

In [ ]:
def peskin_phi(r):
    """Four-point Peskin 1D kernel phi(r), with support |r| <= 2."""
    r = np.abs(np.asarray(r, dtype=float))
    out = np.zeros_like(r)

    m1 = r <= 1.0
    out[m1] = (
        3.0 - 2.0 * r[m1]
        + np.sqrt(np.maximum(0.0, 1.0 + 4.0 * r[m1] - 4.0 * r[m1]**2))
    ) / 8.0

    m2 = (r > 1.0) & (r <= 2.0)
    out[m2] = (
        5.0 - 2.0 * r[m2]
        - np.sqrt(np.maximum(0.0, -7.0 + 12.0 * r[m2] - 4.0 * r[m2]**2))
    ) / 8.0

    return out

r = np.linspace(-2.5, 2.5, 500)
plt.figure(figsize=(7, 3))
plt.plot(r, peskin_phi(r))
plt.xlabel("r")
plt.ylabel("phi(r)")
plt.title("Four-point Peskin regularised delta kernel")
plt.grid(True)
plt.show()

## 5. Spread a marker force and check conservation

In the discrete model, force spreading uses a weighted sum. With marker forces interpreted as force per unit boundary length, a typical form is

```math
\mathbf{F}_{ij}=\sum_k \mathbf{f}_k\,\delta_h(x_i-X_k)\,\delta_h(y_j-Y_k)\,\Delta s_k
```

A useful check is that the grid-integrated force approximately matches the sum of marker forces times their boundary segment lengths:

```math
\sum_{i,j}\mathbf{F}_{ij}\,\Delta x\Delta y
\approx
\sum_k\mathbf{f}_k\,\Delta s_k
```

The exact discrete identity depends on kernel properties, periodic wrapping, grid spacing, and implementation details.

In [ ]:
# Demonstration: spread vertical force density from markers on a periodic horizontal line.
# The marker force values below are illustrative, not physiological.

nx = ny = 64
Lx = Ly = 1.0
dx, dy = Lx / nx, Ly / ny
x = np.arange(nx) * dx
y = np.arange(ny) * dy
Xgrid, Ygrid = np.meshgrid(x, y, indexing="xy")

n_markers = 32
Xm = np.linspace(0.0, Lx, n_markers, endpoint=False)
Ym = np.full(n_markers, 0.5)
ds = Lx / n_markers
f_marker_y = np.ones(n_markers) * 2.0  # N/m, illustrative force density

Fgrid_y = np.zeros((ny, nx))

def periodic_distance(a, b, length):
    """Signed shortest periodic distance from grid coordinate a to marker b."""
    d = a - b
    return (d + 0.5 * length) % length - 0.5 * length

for xm, ym, fm in zip(Xm, Ym, f_marker_y):
    rx = periodic_distance(Xgrid, xm, Lx) / dx
    ry = periodic_distance(Ygrid, ym, Ly) / dy
    delta_x = peskin_phi(rx) / dx
    delta_y = peskin_phi(ry) / dy
    Fgrid_y += fm * delta_x * delta_y * ds

force_from_markers = np.sum(f_marker_y * ds)
force_on_grid = np.sum(Fgrid_y) * dx * dy

print(f"Integrated marker force: {force_from_markers:.8f} N")
print(f"Integrated grid force:   {force_on_grid:.8f} N")
print(f"Absolute difference:     {abs(force_from_markers-force_on_grid):.3e} N")

plt.figure(figsize=(7, 4))
plt.imshow(Fgrid_y, origin="lower", extent=[0, Lx, 0, Ly], aspect="equal")
plt.colorbar(label="Spread vertical force density [N/m²]")
plt.xlabel("x [m]")
plt.ylabel("y [m]")
plt.title("Boundary force spread onto the fixed grid")
plt.show()

## 6. Interpolate velocity to the marker

The interpolation uses the same kernel weights in the reverse direction. This makes the fluid velocity available at marker locations, even when a marker falls between grid points. For a prescribed-motion experiment, compare the interpolated marker velocity with the target membrane velocity. They will not automatically agree unless the coupling method explicitly enforces the constraint.

The next cell creates a smooth test velocity field and interpolates it to the markers. It is a kernel test, not a coupled fluid simulation.

In [ ]:
# Interpolate the projected test velocity to marker points using the same kernel.
U_marker_x = np.zeros(n_markers)
U_marker_y = np.zeros(n_markers)

for k, (xm, ym) in enumerate(zip(Xm, Ym)):
    rx = periodic_distance(Xgrid, xm, Lx) / dx
    ry = periodic_distance(Ygrid, ym, Ly) / dy
    weights = (peskin_phi(rx) / dx) * (peskin_phi(ry) / dy) * dx * dy
    U_marker_x[k] = np.sum(u_new * weights)
    U_marker_y[k] = np.sum(v_new * weights)

print("Interpolated marker velocities (first five markers):")
for k in range(min(5, n_markers)):
    print(f"marker {k:2d}: Ux={U_marker_x[k]: .5e} m/s, Uy={U_marker_y[k]: .5e} m/s")

## 7. Time-step estimates

For an explicit advection discretisation, a common estimate is

```math
\Delta t_{\mathrm{adv}}\leq C_{\mathrm{adv}}\frac{h}{U_{\max}}
```

For forward-Euler diffusion on a two-dimensional square grid,

```math
\Delta t_{\mathrm{visc}}\leq\frac{h^2}{4\nu}
```

where $\nu=\mu/\rho$. These are scheme-dependent estimates. Moving-boundary coupling can impose additional restrictions, especially for stiff penalty forces or elastic markers.

In [ ]:
# Example time-step estimates. Replace these values with the parameters for your case.
U_max = 0.5       # maximum speed [m/s], assumed for estimate
h = 1.0 / 64      # grid spacing [m] for a unit-length domain
nu = 1.0e-6       # kinematic viscosity [m^2/s]
C_adv = 0.5

dt_adv = C_adv * h / U_max
dt_visc = h**2 / (4.0 * nu)
dt_estimate = min(dt_adv, dt_visc)

print(f"Advection estimate: {dt_adv:.4e} s")
print(f"Explicit diffusion estimate: {dt_visc:.4e} s")
print(f"Minimum of these two estimates: {dt_estimate:.4e} s")
print("This is not yet a final time step: check membrane coupling and accuracy too.")

## 8. Implementation plan: from tests to the chamber

Use these milestones as a checklist. Do not add all of them at once.

1. **Fluid solver:** implement momentum advection and viscosity, then pressure projection. Choose and document wall boundary conditions.
2. **Moving-wall benchmark:** drive a simple grid-aligned wall with the prescribed sinusoidal velocity. Check time-step stability and flow response.
3. **Internal membrane markers:** place Lagrangian markers inside the fixed fluid grid and evaluate the target position and velocity each time step.
4. **Coupling:** implement a documented direct-forcing or penalty method so marker velocity approaches the prescribed velocity. Measure the constraint error and test sensitivity to parameters.
5. **Elastic membrane (later):** replace prescribed motion with a force model derived from elastic energy only after prescribed motion works.
6. **Chamber geometry:** add the actual internal membrane and physically consistent outer-wall/inlet/outlet conditions.
7. **Validation:** compare time-step refinements, grid refinements, divergence, force balance, and limiting cases.

## 9. Validation questions

Before treating the result as complete, answer these in the README or a follow-up notebook section:

- Does the projection reduce divergence to the expected tolerance?
- Does the integrated spread force match the marker force within numerical tolerance?
- Does the interpolation behave smoothly as a marker moves across a grid cell?
- Does halving the time step change the measured flow response significantly?
- Does refining the grid change the result in a way consistent with convergence?
- Is the motion prescribed, or is the membrane genuinely responding to fluid and elastic forces?

## Limitations

The cells in this notebook are small numerical experiments and teaching scaffolds. They do not yet form a complete moving-wall or immersed-boundary Navier–Stokes solver, and they should not be presented as a validated simulation of a biological heart. The next implementation task is to build the actual fluid time step and wall/coupling treatment, with explicit tests for each part.

## Reference

Peskin, C. S. (1977). *Numerical Analysis of Blood Flow in the Heart*. Journal of Computational Physics, 25(3), 220–252. [https://doi.org/10.1016/0021-9991(77)90100-0](https://doi.org/10.1016/0021-9991(77)90100-0).